# SPFX-EXTRACTOR V1.21
---
**INSTRUCTIONS :**
1. Il faut prévoir un court délai entre l'upload sur le drive et la sync des fichiers sur l'environnement colab.
2. Activez le GPU : Exécution > Modifier le type d'exécution > **GPU T4**.
3. Ajustez votre niveau de réduction de bruit dans la Cellule 1.
4. Exécutez toutes les cellules **dans l'ordre**.


**⚠️ NOTE IMPORTANTE SUR L'AUTO-ALIGNEMENT INITIAL :**
En TOUT PREMIER LIEU, avant d'appliquer quelque processing que ce soit, le script regroupe les clips par numéro de prise. Il va ensuite s'assurer d'auto-aligner les clips ensemble (ex: `Lav-01.wav` va s'autoaligner sur `Boom-01.wav`).
Si les clips d'un même groupe ne contiennent pas spécifiquement les mots "Lav" et "Boom" dans leur nom, la piste de référence (la source de cet auto-alignement) sera choisie de façon aléatoire parmi le groupe.

**NOUVEAUTÉS V1.21 :**
- **Hard Purge :** Nettoyage agressif des dossiers temporaires au lancement pour éviter le syndrome des "Fichiers Fantômes" (fichiers résiduels des anciens essais qui by-passent les nouvelles fusions).

In [1]:
# 1. 🎛️ PANNEAU DE CONTRÔLE UTILISATEUR
#@title Ajustement de la Réduction de Bruit (Pre-Denoise)
#@markdown Glissez le curseur pour définir la force du nettoyage avant le passage de l'IA.
#@markdown * 45% = Standard (Préserve la dynamique et la "guenille")
#@markdown * 80%+ = Agressif (Pour les scènes d'autoroute ou industrielles)

NIVEAU_DENOISE_POURCENTAGE = 58 #@param {type:"slider", min:0, max:100, step:1}

print(f"✅ Niveau de réduction de bruit réglé à : {NIVEAU_DENOISE_POURCENTAGE}%")

✅ Niveau de réduction de bruit réglé à : 58%


In [2]:
# 2. INSTALLATION DES DÉPENDANCES ET VÉRIFICATION DU GPU
print("🚀 Installation de audio-separator et des dépendances...")
!pip install -q "audio-separator[gpu]" soundfile scipy noisereduce
!pip uninstall -y onnxruntime onnxruntime-gpu
!pip install -q onnxruntime-gpu --extra-index-url https://aiinfra.pkgs.visualstudio.com/PublicPackages/_packaging/onnxruntime-cuda-12/pypi/simple/
from IPython.display import clear_output
clear_output()
print("✅ Dépendances installées.")

import torch
device = 'cuda' if torch.cuda.is_available() else 'cpu'
print(f"🖥️  Device : {device.upper()}")
if device == 'cpu':
    print("   ⚠️  Aucun GPU — activez le GPU T4 dans Exécution > Modifier le type d'exécution.")
else:
    print(f"   ✅ Carte graphique détectée : {torch.cuda.get_device_name(0)}")

✅ Dépendances installées.
🖥️  Device : CUDA
   ✅ Carte graphique détectée : Tesla T4


In [3]:
# 3. MONTAGE DE GOOGLE DRIVE ET DOSSIERS TEMP
import os
from google.colab import drive
drive.mount('/content/drive')

DRIVE_BASE = "/content/drive/MyDrive/PFX_Extractor"
FOLDER_IN       = os.path.join(DRIVE_BASE, "1_Bruts_vers_Colab")
FOLDER_OUT      = os.path.join(DRIVE_BASE, "2_Environnements_IA")

FOLDER_ALIGNED  = "/content/temp_aligned"
FOLDER_MERGED   = "/content/temp_merged"
FOLDER_DENOISED = "/content/temp_denoised"
FOLDER_AI_STEMS = "/content/temp_ai_stems"
FOLDER_TC_READY = "/content/temp_tc_ready"

for folder in [FOLDER_IN, FOLDER_OUT, FOLDER_ALIGNED, FOLDER_MERGED, FOLDER_DENOISED, FOLDER_AI_STEMS, FOLDER_TC_READY]:
    os.makedirs(folder, exist_ok=True)

print(f"\n📁 Dossiers configurés.")

Mounted at /content/drive

📁 Dossiers configurés.


In [4]:
# 4. CONFIGURATION DES MODÈLES
import os
import gc
import torch
import inspect
import shutil
import numpy as np
import soundfile as sf
import noisereduce as nr
import subprocess
import re
from scipy import signal
from audio_separator.separator import Separator

os.makedirs('/content/models',   exist_ok=True)
os.makedirs('/content/temp_out', exist_ok=True)

MODEL_MDX23C = 'MDX23C-8KFFT-InstVoc_HQ.ckpt'
MODEL_ROFORMER = 'model_bs_roformer_ep_317_sdr_12.9755.ckpt'

MDXC_PARAMS = {'segment_size': 256, 'batch_size': 1, 'overlap': 8}
ROFORMER_PARAMS = {'segment_size': 256, 'override_model_segment_size': True, 'batch_size': 1, 'num_overlap': 8}

TEMP_IN  = "/content/temp_in.wav"
TEMP_OUT = "/content/temp_out"
MIN_SAFE_DURATION = 5.0

print("✅ Configuration terminée.")

ImportError: libcudart.so.13: cannot open shared object file: No such file or directory

In [ ]:
# 5. MOTEUR DSP, TIMECODE ET ALGORITHMES STRUCTURELS
def natural_sort_key(s):
    return [int(text) if text.isdigit() else text.lower() for text in re.split(r'(\d+)', s)]

def initial_mic_alignment(files_list):
    print(f"\n{'='*50}")
    print(f"🎯 ÉTAPE 0.1 : AUTO-ALIGNEMENT INITIAL (LAV -> BOOM)")
    print(f"{'='*50}")

    take_groups = {}
    for f in files_list:
        nums = re.findall(r'\d+', f)
        take = nums[-1] if nums else f
        if take not in take_groups: take_groups[take] = []
        take_groups[take].append(f)

    aligned_files = []
    for take, files in take_groups.items():
        if len(files) == 1:
            shutil.copy(os.path.join(FOLDER_IN, files[0]), os.path.join(FOLDER_ALIGNED, files[0]))
            aligned_files.append(files[0])
            continue

        ref_file = next((x for x in files if 'boom' in x.lower()), files[0])
        ref_path = os.path.join(FOLDER_IN, ref_file)
        shutil.copy(ref_path, os.path.join(FOLDER_ALIGNED, ref_file))
        aligned_files.append(ref_file)

        print(f"   🎙️ Groupe [Take {take}] : Référence -> {ref_file}")

        ref_audio, sr = sf.read(ref_path, always_2d=True)
        ref_mono = np.mean(ref_audio, axis=1)
        max_samples = 30 * sr

        for tgt_file in files:
            if tgt_file == ref_file: continue
            print(f"      🧲 Alignement de la cible : {tgt_file}")
            tgt_path = os.path.join(FOLDER_IN, tgt_file)
            tgt_audio, _ = sf.read(tgt_path, always_2d=True)
            tgt_mono = np.mean(tgt_audio, axis=1)

            if np.max(np.abs(ref_mono)) < 1e-6 or np.max(np.abs(tgt_mono)) < 1e-6:
                sf.write(os.path.join(FOLDER_ALIGNED, tgt_file), tgt_audio, sr, subtype='FLOAT')
                aligned_files.append(tgt_file)
                continue

            corr = signal.correlate(ref_mono[:max_samples], tgt_mono[:max_samples], mode='full', method='fft')
            lag = np.argmax(corr) - (len(tgt_mono[:max_samples]) - 1)

            tgt_aligned = np.zeros_like(tgt_audio)
            if lag > 0:
                length = min(len(ref_audio), len(tgt_audio) - lag)
                tgt_aligned[:length] = tgt_audio[lag:lag+length]
            elif lag < 0:
                lag = abs(lag)
                length = min(len(ref_audio) - lag, len(tgt_audio))
                tgt_aligned[lag:lag+length] = tgt_audio[:length]
            else:
                tgt_aligned = tgt_audio[:len(ref_audio)]

            sf.write(os.path.join(FOLDER_ALIGNED, tgt_file), tgt_aligned, sr, subtype='FLOAT')
            aligned_files.append(tgt_file)

    return aligned_files

def snowball_merge(files_list):
    print(f"\n{'='*50}")
    print(f"☃️  ÉTAPE 0.2 : SNOWBALL MERGE (Séparation par Famille)")
    print(f"{'='*50}")

    families = {}
    for f in files_list:
        match = re.search(r'^(.*)([-_ ]\d+)', f)
        base_name = match.group(1).strip() if match else "Misc"
        if base_name not in families: families[base_name] = []
        families[base_name].append(f)

    merged_files_map = {}

    for base_name, fam_files in families.items():
        fam_files = sorted(fam_files, key=natural_sort_key)
        current_group_audio = None
        current_sr = None
        anchor_filename = None
        current_duration = 0.0

        family_anchors = []

        for f in fam_files:
            in_path = os.path.join(FOLDER_ALIGNED, f)
            audio, sr = sf.read(in_path, always_2d=True)
            duration = audio.shape[0] / sr

            if current_group_audio is None:
                current_group_audio = audio
                current_sr = sr
                anchor_filename = f
                current_duration = duration
            else:
                current_group_audio = np.concatenate((current_group_audio, audio), axis=0)
                current_duration += duration

            if current_duration >= MIN_SAFE_DURATION:
                out_path = os.path.join(FOLDER_MERGED, anchor_filename)
                sf.write(out_path, current_group_audio, current_sr, subtype='FLOAT')
                merged_files_map[anchor_filename] = anchor_filename
                family_anchors.append(anchor_filename)
                if current_duration != duration:
                    print(f"   🔗 Fusion Famille [{base_name}] : '{anchor_filename}' cuit à {current_duration:.2f}s")
                current_group_audio = None
                current_duration = 0.0

        if current_group_audio is not None:
            if not family_anchors:
                 out_path = os.path.join(FOLDER_MERGED, anchor_filename)
                 sf.write(out_path, current_group_audio, current_sr, subtype='FLOAT')
                 merged_files_map[anchor_filename] = anchor_filename
            else:
                 last_anchor = family_anchors[-1]
                 last_path = os.path.join(FOLDER_MERGED, last_anchor)
                 last_audio, _ = sf.read(last_path, always_2d=True)
                 combined = np.concatenate((last_audio, current_group_audio), axis=0)
                 sf.write(last_path, combined, current_sr, subtype='FLOAT')
                 print(f"   🔗 Fusion finale Famille [{base_name}] : Reste greffé à '{last_anchor}'")

    return list(merged_files_map.keys()), merged_files_map

def get_sample_rate(filepath):
    try:
        _, sr = sf.read(filepath, frames=1)
        return sr
    except: return 48000

def inject_timecode(original_path, processed_path, final_output_path):
    original_sr = get_sample_rate(original_path)
    try:
        probe_cmd = ['ffprobe', '-v', 'error', '-show_entries', 'format_tags=time_reference', '-of', 'default=nw=1:nk=1', original_path]
        time_ref = subprocess.check_output(probe_cmd).decode('utf-8').strip()
    except Exception: time_ref = ""

    cmd = ['ffmpeg', '-y', '-loglevel', 'error', '-i', processed_path, '-i', original_path, '-map', '0:a', '-map_metadata', '1', '-write_bext', '1']
    if time_ref: cmd.extend(['-metadata:g', f'time_reference={time_ref}'])
    cmd.extend(['-ar', str(original_sr), '-c:a', 'pcm_f32le', final_output_path])
    subprocess.run(cmd)

def prep_denoise_batch(files_list):
    print(f"\n{'='*50}\n🧹 ÉTAPE 1 : PRÉ-NETTOYAGE OBLIGATOIRE ({NIVEAU_DENOISE_POURCENTAGE}%)\n{'='*50}")

    prop_decrease_val = NIVEAU_DENOISE_POURCENTAGE / 100.0

    for f in files_list:
        in_path = os.path.join(FOLDER_MERGED, f)
        out_path = os.path.join(FOLDER_DENOISED, f)
        # Suppression du "if os.path.exists" pour s'assurer d'écraser les vieux fichiers

        print(f"   🚿 Nettoyage chirurgical ({NIVEAU_DENOISE_POURCENTAGE}%) : {f}")
        audio, sr = sf.read(in_path, always_2d=True)
        reduced_audio = np.zeros_like(audio)
        for i in range(audio.shape[1]):
            reduced_channel = nr.reduce_noise(
                y=audio[:, i], sr=sr, stationary=True, prop_decrease=prop_decrease_val,
                n_fft=8192, win_length=8192, hop_length=512,
                time_mask_smooth_ms=150, freq_mask_smooth_hz=500
            )
            if np.isnan(reduced_channel).any(): reduced_channel = audio[:, i]
            reduced_audio[:, i] = reduced_channel
        sf.write(out_path, reduced_audio, sr, subtype='FLOAT')

def _cleanup_temp():
    for fname in os.listdir(TEMP_OUT):
        fpath = os.path.join(TEMP_OUT, fname)
        if os.path.isfile(fpath):
            try: os.remove(fpath)
            except: pass
    if os.path.exists(TEMP_IN): os.remove(TEMP_IN)

def process_batch(model_name, model_suffix, custom_params, files_list):
    if not files_list: return
    print(f"\n{'='*50}\n⚙️  CHARGEMENT DU MODÈLE : {model_name}\n{'='*50}")
    proposed_kwargs = {'log_level': 30, 'model_file_dir': '/content/models', 'output_dir': '/content/temp_out', 'output_format': 'WAV', 'normalization_threshold': 1.0, 'output_single_stem': 'Instrumental'}
    if 'MDX23C' in model_name: proposed_kwargs['mdxc_params'] = custom_params
    elif 'roformer' in model_name.lower(): proposed_kwargs['roformer_params'] = custom_params
    sig = inspect.signature(Separator.__init__)
    safe_kwargs = {k: v for k, v in proposed_kwargs.items() if k in set(sig.parameters.keys())}
    separator = Separator(**safe_kwargs)
    separator.load_model(model_name)

    for f in files_list:
        input_path = os.path.join(FOLDER_DENOISED, f)
        base, _ = os.path.splitext(f)
        output_path = os.path.join(FOLDER_AI_STEMS, base + model_suffix)
        # Suppression du by-pass pour forcer la réécriture de l'IA

        print(f"\n   🎵 Traitement : {f}")
        audio, sr = sf.read(input_path, always_2d=True)
        audio_stereo = np.repeat(audio, 2, axis=1) if (audio.shape[1] == 1) else audio
        sf.write(TEMP_IN, audio_stereo, sr, subtype='FLOAT')

        try: res_files = separator.separate(TEMP_IN)
        except Exception as e: res_files = None

        if not res_files:
            print(f"   🔄 FALLBACK : Bypass de l'IA.")
            sf.write(output_path, audio, sr, subtype='FLOAT')
            _cleanup_temp()
            continue

        inst_path = os.path.join(TEMP_OUT, res_files[0]) if not os.path.isabs(res_files[0]) else res_files[0]
        if not os.path.exists(inst_path):
            print(f"   🔄 FALLBACK : Fichier silencieux (ignoré par l'IA).")
            sf.write(output_path, audio, sr, subtype='FLOAT')
            _cleanup_temp()
            continue
        inst_audio, inst_sr = sf.read(inst_path, always_2d=True)
        if audio.shape[1] == 1: inst_audio = np.mean(inst_audio, axis=1, keepdims=True)

        sf.write(output_path, inst_audio, inst_sr, subtype='FLOAT')
        _cleanup_temp()

    del separator
    gc.collect()
    torch.cuda.empty_cache()

def auto_align_and_mix(ref_path, target_path, output_path):
    ref_audio, sr = sf.read(ref_path, always_2d=True)
    tgt_audio, _ = sf.read(target_path, always_2d=True)
    ref_mono = np.mean(ref_audio, axis=1)
    tgt_mono = np.mean(tgt_audio, axis=1)
    max_samples = 30 * sr

    if np.max(np.abs(ref_mono)) < 1e-6 or np.max(np.abs(tgt_mono)) < 1e-6:
        sf.write(output_path, (ref_audio + tgt_audio) / 2.0, sr, subtype='FLOAT')
        return

    corr = signal.correlate(ref_mono[:max_samples], tgt_mono[:max_samples], mode='full', method='fft')
    lag = np.argmax(corr) - (len(tgt_mono[:max_samples]) - 1)

    tgt_aligned = np.zeros_like(ref_audio)
    if lag > 0:
        length = min(len(ref_audio), len(tgt_audio) - lag)
        tgt_aligned[:length] = tgt_audio[lag:lag+length]
    elif lag < 0:
        lag = abs(lag)
        length = min(len(ref_audio) - lag, len(tgt_audio))
        tgt_aligned[lag:lag+length] = tgt_audio[:length]
    else:
        tgt_aligned = tgt_audio[:len(ref_audio)]

    sf.write(output_path, (ref_audio + tgt_aligned) / 2.0, sr, subtype='FLOAT')

In [ ]:
# 6. LANCEMENT DU WORKFLOW COMPLET INTELLIGENT
print("\n🧹 PURGE DES DOSSIERS TEMPORAIRES (Prévention des fichiers fantômes)...")
for folder in [FOLDER_ALIGNED, FOLDER_MERGED, FOLDER_DENOISED, FOLDER_AI_STEMS, FOLDER_TC_READY]:
    if os.path.exists(folder):
        for tmp_f in os.listdir(folder):
            try: os.remove(os.path.join(folder, tmp_f))
            except: pass

all_files_in = [f for f in os.listdir(FOLDER_IN) if f.lower().endswith('.wav')]

if not all_files_in:
    print("\n📭 Aucun fichier .wav trouvé dans le dossier IN.")
else:
    print(f"\n🔥 {len(all_files_in)} fichier(s) détecté(s). Début du workflow...\n")

    aligned_files_list = initial_mic_alignment(all_files_in)
    merged_files_list, merged_map = snowball_merge(aligned_files_list)

    prep_denoise_batch(merged_files_list)

    process_batch(MODEL_MDX23C, "_MDX23C.wav", MDXC_PARAMS, merged_files_list)
    process_batch(MODEL_ROFORMER, "_RoFormer.wav", ROFORMER_PARAMS, merged_files_list)

    print(f"\n{'='*50}\n🎛️  ÉTAPE 4 : AUTO-ALIGNEMENT IA ET ANCRAGE TC BWF\n{'='*50}\n")

    for f in merged_files_list:
        base, _ = os.path.splitext(f)
        mdx_path = os.path.join(FOLDER_AI_STEMS, base + "_MDX23C.wav")
        rof_path = os.path.join(FOLDER_AI_STEMS, base + "_RoFormer.wav")
        mix_path = os.path.join(FOLDER_TC_READY, base + "_Mix_Temp.wav")
        final_out_path = os.path.join(FOLDER_OUT, base + "_PFX_Ready.wav")

        original_anchor_filename = merged_map[f]
        original_in_path = os.path.join(FOLDER_IN, original_anchor_filename)

        # Suppression du by-pass de sécurité final pour forcer le Timecode
        if os.path.exists(mdx_path) and os.path.exists(rof_path):
             auto_align_and_mix(rof_path, mdx_path, mix_path)

             if os.path.exists(mix_path):
                 print(f"   ⏱️ Ancrage du TC (Source: {original_anchor_filename}) pour : {base}")
                 inject_timecode(original_in_path, mix_path, final_out_path)
                 print(f"      ✅ Prêt : {os.path.basename(final_out_path)}")

    print("\n🧹 Nettoyage des dossiers temporaires...")
    for folder in [FOLDER_ALIGNED, FOLDER_MERGED, FOLDER_DENOISED, FOLDER_AI_STEMS, FOLDER_TC_READY]:
        for tmp_f in os.listdir(folder):
            os.remove(os.path.join(folder, tmp_f))

    print("\n🎉 WORKFLOW CONTINU TERMINÉ !")

In [ ]:
# 7. DÉCONNEXION AUTOMATIQUE
import time
import torch
from google.colab import runtime

# --- CALCUL DU COÛT DE LA SESSION ---
try:
    uptime_seconds = float(open('/proc/uptime').read().split()[0])
    uptime_hours = uptime_seconds / 3600.0
    gpu_name = torch.cuda.get_device_name(0)

    # Taux approximatifs de Compute Units par heure sur Colab
    rates = {'T4': 1.96, 'L4': 5.09, 'V100': 5.4, 'A100': 13.08}
    rate = 1.96 # defaut
    for k, v in rates.items():
        if k in gpu_name: rate = v; break

    units_used = uptime_hours * rate
    cost_usd = units_used * (16.0 / 100.0) # 16$ par 100 units

    print("\n" + "="*40)
    print("💰 BILAN FINANCIER DE LA SESSION")
    print("="*40)
    print(f"⏱️ Temps d'exécution total : {uptime_hours:.2f} heures")
    print(f"🖥️ GPU Utilisé : {gpu_name} ({rate} unités/heure)")
    print(f"⚡ Compute Units brûlés : ~{units_used:.2f}")
    print(f"💸 Coût estimé : {cost_usd:.3f} USD")
    print("="*40 + "\n")
except Exception as e:
    print(f"\n⚠️ Impossible de calculer le coût : {e}")

print("⏳ L'instance Colab se déconnectera automatiquement dans 10 secondes...")
time.sleep(10)
print("👋 Déconnexion. Au revoir!")
runtime.unassign()
